# 1. Experiment purpose and hybrid Forecast formula

This notebook compares fixed baseline candidates for predicting the correction to the deterministic forecast. It does not select a model automatically.

```text
residual_target = actual_remaining_net_flow - deterministic_remaining_net_flow
predicted_remaining_net_flow = deterministic_remaining_net_flow + predicted_residual
```

The mandatory deterministic reference predicts `predicted_residual = 0`. The naive reference predicts `predicted_remaining_net_flow = 0`; these are different references. MAE is the primary selection metric and RMSE is the secondary, shock-sensitive metric. Full-dataset EDA figures are descriptive and must not drive selection.

# 2. Imports and reproducibility settings

In [ ]:
import os
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

from forecast_ml.evaluation import (
    evaluation_table,
    prediction_frame,
    reference_comparison,
    residual_zero_predictions,
)
from forecast_ml.feature_matrix import (
    FEATURE_COLUMNS,
    FEATURE_CONTRACT_VERSION,
    FORBIDDEN_FEATURE_COLUMNS,
    build_experiment_matrix,
    combine_experiment_matrices,
    load_processed_dataset,
)
from forecast_ml.modeling import (
    CANDIDATE_MODEL_NAMES,
    create_candidate_pipeline,
    create_candidate_pipelines,
    candidate_configuration,
    linear_coefficients,
)
from forecast_ml.temporal_split import (
    MonthRange,
    TemporalBoundaries,
    chronological_split,
)

# 3. Configuration

Do not adjust boundaries silently to fit the data. If several currencies exist, choose exactly one. Leave the final-test controls locked until the validation decision is written down.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Run Jupyter from the ml-pipeline directory')

PROCESSED_DATASET_PATH = Path(os.getenv(
    'FORECAST_PROCESSED_DATA',
    'data/processed/forecast_residual_examples.csv',
))
if not PROCESSED_DATASET_PATH.is_absolute():
    PROCESSED_DATASET_PATH = PROJECT_ROOT / PROCESSED_DATASET_PATH

EXPERIMENT_CURRENCY = os.getenv('FORECAST_EXPERIMENT_CURRENCY', 'TND')
RANDOM_SEED = 20260818
BOUNDARIES = TemporalBoundaries(
    training=MonthRange('2024-01', '2025-06'),
    validation=MonthRange('2025-07', '2025-09'),
    test=MonthRange('2025-10', '2025-12'),
)

# Edit these only after recording the validation-based decision below.
SELECTED_MODEL_NAME = "huber"
MODEL_SELECTION_FROZEN = True
RUN_PERMUTATION_IMPORTANCE = False

np.random.seed(RANDOM_SEED)
print({
    'dataset': str(PROCESSED_DATASET_PATH),
    'currency': EXPERIMENT_CURRENCY,
    'seed': RANDOM_SEED,
    'feature_contract_version': FEATURE_CONTRACT_VERSION,
    'candidate_order': CANDIDATE_MODEL_NAMES,
})

# 4. Load and validate the processed dataset

The loader preserves strings and scale-4 decimals until `build_experiment_matrix`, the explicit scikit-learn float boundary.

In [ ]:
processed = load_processed_dataset(PROCESSED_DATASET_PATH)
print(f'Validated processed rows: {len(processed):,}')
display(processed.head())

# 5. Profile, month, cutoff, and fallback coverage

Five cutoff rows from one profile-month are correlated observations. For example, 600 rows represent 120 independent profile-month groups, not 600 independent months.

In [ ]:
coverage = pd.Series({
    'rows': len(processed),
    'profiles': processed['profile_key'].nunique(),
    'currencies': processed['currency'].nunique(),
    'target_months': processed['target_month'].nunique(),
    'independent_profile_month_groups': len(
        processed[['profile_key', 'currency', 'target_month']].drop_duplicates()
    ),
})
display(coverage.to_frame('value'))
display(processed['cutoff_day'].value_counts().sort_index().rename('rows').to_frame())
display(processed['deterministic_fallback_reason'].value_counts().rename('rows').to_frame())

# 6. Build the explicit leakage-safe feature matrix

Calendar month is encoded as sine/cosine on a 12-month cycle, so December and January remain adjacent. Raw month/date, identity, currency, version, target, and realized outcomes are never predictors.

In [ ]:
experiment = build_experiment_matrix(processed, currency=EXPERIMENT_CURRENCY)
assert not FORBIDDEN_FEATURE_COLUMNS.intersection(experiment.features.columns)
print(f'Experiment rows for {experiment.currency}: {len(experiment.features):,}')
display(experiment.features.head())

# 7. Final permitted feature names

This list is a candidate contract, not a final feature-selection decision.

In [ ]:
display(pd.DataFrame({'permitted_feature': FEATURE_COLUMNS}))
display(pd.DataFrame({'forbidden_feature': sorted(FORBIDDEN_FEATURE_COLUMNS)}))

# 8. Chronological train/validation/test partitions

Splitting uses `target_month`, never row order or randomness. Preprocessing remains inside each model pipeline and is fitted only with training rows. Test rows and labels are held by a guard.

In [ ]:
partitions = chronological_split(experiment, BOUNDARIES)
training = partitions.training
validation = partitions.validation
display(partitions.partition_summary())

# 9. Partition sizes and independent groups

Only aggregate test size is visible. Do not attempt to inspect private guard state.

In [ ]:
for name, matrix in [('training', training), ('validation', validation)]:
    month_range = matrix.context['target_month']
    groups = matrix.context[['profile_key', 'currency', 'target_month']].drop_duplicates()
    print(name, {
        'rows': len(matrix.features),
        'independent_profile_month_groups': len(groups),
        'first_month': str(month_range.min()),
        'last_month': str(month_range.max()),
    })

# 10. Naive and deterministic validation references

The residual-zero deterministic reference retains the deterministic remaining-flow estimate. The naive reference instead predicts zero remaining flow. Cutoff 25 is always displayed separately because the EDA indicated a possible horizon-specific weakness; only validation evidence may influence the later decision.

In [ ]:
display(reference_comparison(validation))
deterministic_residual = residual_zero_predictions(len(validation.features))
naive_residual = -validation.context['deterministic_remaining_net_flow'].to_numpy(dtype=float)
deterministic_by_cutoff = evaluation_table(
    validation, deterministic_residual, group_by='cutoff_day'
).assign(reference='residual_zero_deterministic')
naive_by_cutoff = evaluation_table(
    validation, naive_residual, group_by='cutoff_day'
).assign(reference='naive_zero_remaining_flow')
reference_by_cutoff = pd.concat(
    [deterministic_by_cutoff, naive_by_cutoff], ignore_index=True
)
display(reference_by_cutoff)
display(reference_by_cutoff.loc[reference_by_cutoff['cutoff_day'] == 25])

# 11. Instantiate fixed candidate pipelines

No hyperparameter search or winner-selection routine is used. Ridge and Huber scale numeric inputs; all learned preprocessing stays inside the training pipeline. Fallback reason is one-hot encoded explicitly.

In [ ]:
candidate_pipelines = create_candidate_pipelines(random_seed=RANDOM_SEED)
display(candidate_configuration(candidate_pipelines))

# 12. Fit candidates on training data

Run this cell yourself. It fits only the training partition and predicts only validation rows.

In [ ]:
validation_predictions = {}
for model_name in CANDIDATE_MODEL_NAMES:
    pipeline = candidate_pipelines[model_name]
    pipeline.fit(training.features, training.target)
    validation_predictions[model_name] = pipeline.predict(validation.features)
print('Training complete for fixed candidates; no model has been selected.')

# 13. Compare validation metrics

Tables remain in the fixed candidate order. Do not choose solely from one aggregate number. Per-cutoff, history, fallback, and profile diagnostics follow. Profile results are diagnostic only.

In [ ]:
overall_tables = []
cutoff_tables = []
history_tables = []
fallback_tables = []
profile_tables = []
for model_name in CANDIDATE_MODEL_NAMES:
    predicted = validation_predictions[model_name]
    overall_tables.append(evaluation_table(validation, predicted).assign(model=model_name))
    cutoff_tables.append(evaluation_table(validation, predicted, group_by='cutoff_day').assign(model=model_name))
    history_tables.append(evaluation_table(validation, predicted, group_by='usable_history_count').assign(model=model_name))
    fallback_tables.append(evaluation_table(validation, predicted, group_by='deterministic_fallback_reason').assign(model=model_name))
    profile_tables.append(evaluation_table(validation, predicted, group_by='profile_key').assign(model=model_name))

validation_overall = pd.concat(overall_tables, ignore_index=True)
validation_by_cutoff = pd.concat(cutoff_tables, ignore_index=True)
validation_by_history = pd.concat(history_tables, ignore_index=True)
validation_by_fallback = pd.concat(fallback_tables, ignore_index=True)
validation_by_profile = pd.concat(profile_tables, ignore_index=True)
display(validation_overall)
display(validation_by_history)
display(validation_by_fallback)
display(validation_by_profile)

# 14. Validation MAE by model

This chart supports review; it does not select the smallest bar automatically.

In [ ]:
fig, axis = plt.subplots(figsize=(9, 4))
axis.bar(validation_overall['model'], validation_overall['mae'], color='#1687F8')
axis.set_ylabel('Validation MAE')
axis.set_title('Overall validation MAE — fixed candidate order')
axis.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()

# 15. Validation MAE by cutoff, including day 25

No production gate is encoded here. Record later whether validation supports all cutoffs, selected cutoffs, deterministic-only behavior, or future horizon-aware shrinkage.

In [ ]:
cutoff_pivot = validation_by_cutoff.pivot(index='cutoff_day', columns='model', values='mae')
display(validation_by_cutoff)
display(validation_by_cutoff.loc[validation_by_cutoff['cutoff_day'] == 25])
axis = cutoff_pivot.plot(kind='bar', figsize=(11, 5))
axis.set_ylabel('Validation MAE')
axis.set_title('Validation MAE by prediction cutoff')
axis.legend(title='Candidate')
plt.tight_layout()
plt.show()

# 16. Residual-prediction errors

Outliers remain visible; no row is deleted. Error is `predicted_residual - residual_target`.

In [ ]:
residual_errors = pd.DataFrame({
    model_name: validation_predictions[model_name] - validation.target.to_numpy()
    for model_name in CANDIDATE_MODEL_NAMES
})
residual_errors.plot.box(figsize=(11, 5), showfliers=True)
plt.axhline(0.0, color='black', linewidth=1)
plt.ylabel('Residual-prediction error')
plt.title('Validation residual errors — outliers retained')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 17. Predicted versus actual remaining net flow

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
for model_name, axis in zip(CANDIDATE_MODEL_NAMES, axes.flat):
    predictions = prediction_frame(validation, validation_predictions[model_name])
    actual = predictions['actual_remaining_net_flow']
    predicted = predictions['predicted_remaining_net_flow']
    minimum = min(actual.min(), predicted.min())
    maximum = max(actual.max(), predicted.max())
    axis.scatter(actual, predicted, alpha=0.65, color='#1687F8')
    axis.plot([minimum, maximum], [minimum, maximum], color='#F3685B', linewidth=1)
    axis.set_title(model_name)
    axis.set_xlabel('Actual remaining net flow')
    axis.set_ylabel('Predicted remaining net flow')
fig.tight_layout()
plt.show()

# 18. Optional validation-only feature interpretation

Linear coefficients are shown where meaningful. Optional permutation importance uses validation data only and is disabled by default because repeated inspection can itself overfit decisions. Importance indicates association with predictive performance, **not causality**.

In [ ]:
for model_name in ('ridge', 'huber'):
    print(model_name)
    display(linear_coefficients(candidate_pipelines[model_name]))

if RUN_PERMUTATION_IMPORTANCE:
    for model_name in CANDIDATE_MODEL_NAMES:
        result = permutation_importance(
            candidate_pipelines[model_name],
            validation.features,
            validation.target,
            scoring='neg_mean_absolute_error',
            n_repeats=20,
            random_state=RANDOM_SEED,
        )
        importance = pd.DataFrame({
            'feature': validation.features.columns,
            'mean_validation_importance': result.importances_mean,
            'standard_deviation': result.importances_std,
        }).sort_values('mean_validation_importance', ascending=False)
        print(model_name, '— importance is not causality')
        display(importance)
else:
    print('Permutation importance not run. Set RUN_PERMUTATION_IMPORTANCE=True only for an intentional validation analysis.')

# 19. Manual model-selection record

Complete every TODO **before** changing the final-test guard.

- TODO — Features retained and rejected, with reasons:
- TODO — Validation results (MAE primary, RMSE secondary):
- TODO — Model selected:
- TODO — Why it was selected rather than simply taking the smallest aggregate MAE:
- TODO — Stability across cutoff days:
- TODO — Whether fallback cases require separate future behavior:
- TODO — Outlier treatment (none are deleted in this notebook):
- TODO — Day-25 decision: all cutoffs / selected cutoffs / deterministic-only / later horizon-aware shrinkage:
- TODO — Known limitations of synthetic data:
- TODO — Acceptable for later real anonymized-data evaluation?
- TODO — Confirm explicitly that this decision was frozen before any test results were viewed:

After completing this record, edit the configuration cell: set `SELECTED_MODEL_NAME` to one exact candidate and `MODEL_SELECTION_FROZEN = True`.

# 20. Guarded final-test evaluation

The next cell refuses to reveal test rows unless the selected name is known and the decision is exactly frozen. Once released, it recreates a fresh pipeline, fits combined training+validation data, evaluates test exactly once, and writes no artifact. Do not revise the choice from these results.

In [ ]:
final_test = partitions.release_test_set(
    selected_model_name=SELECTED_MODEL_NAME,
    model_selection_frozen=MODEL_SELECTION_FROZEN,
    known_model_names=CANDIDATE_MODEL_NAMES,
)
development = combine_experiment_matrices(training, validation)
final_pipeline = create_candidate_pipeline(
    SELECTED_MODEL_NAME, random_seed=RANDOM_SEED
)
final_pipeline.fit(development.features, development.target)
final_test_predictions = final_pipeline.predict(final_test.features)

print('Frozen model:', SELECTED_MODEL_NAME)
display(reference_comparison(final_test))
display(evaluation_table(final_test, final_test_predictions).assign(model=SELECTED_MODEL_NAME))
display(evaluation_table(
    final_test, final_test_predictions, group_by='cutoff_day'
).assign(model=SELECTED_MODEL_NAME))
print('Final test evaluated once. No model artifact was persisted; do not revise selection from these results.')

# 21. Final experiment conclusions and limitations

Complete only after the guarded test has been legitimately unlocked.

- TODO — Frozen validation decision and timestamp:
- TODO — Untouched test result summary, reported once:
- TODO — Differences between validation and test behavior:
- TODO — Day-25 observation without adding a production gate:
- TODO — Synthetic-data limitations and real anonymized-data validation required next:
- TODO — Decision on whether further experimentation is justified:

Synthetic-data performance does not demonstrate production accuracy. This notebook persists no trained model and makes no deployment decision. Clear all outputs before committing the notebook source.


Final conclusions to record
Frozen model: Huber.
Test MAE improvement: 4.89% over deterministic.
Test RMSE improvement: approximately 1%.
Median error improved substantially, but financial shocks remain poorly predicted.
Huber became more pessimistic, underpredicting remaining net flow by approximately 168.56 TND on average.
Performance improved for cutoffs 5–20.
Day-25 behavior was unstable between validation and test.
Model selection will not be revised using test results.
Fallback behavior remains unvalidated because validation and test contain only full-history rows.
The experiment uses synthetic TND data from five profiles and does not demonstrate production accuracy.
Huber is acceptable as a research baseline for real anonymized-data evaluation, not as a deployable model.